# Beyond the Score
## Chapter 4 — Reproduction of reported results

This notebook follows the experiments reported in **Beyond the Score: Validating Evaluations of Synthetic IoT Time Series**. It presents the metric-response experiments, residential task qualification and utility, constructed benchmark, and external TON_IoT evaluation.

The final comparison covers **45 registered headline values**, all rows of the selected stage comparisons, and their saved-prediction verification. The coverage map is in `CHAPTER_COVERAGE.csv`. Later exploratory evaluations are outside this notebook's scope.

**Two execution modes:** `evidence` verifies the supplied completed author-run results and recalculates their headline values. `fresh` executes the chapter experiments from the specified inputs. Every table states which mode is being used. A successful evidence review is not a new model-training run.


## 1. Select the execution mode and locations

For a first review, leave `MODE = "evidence"`. For a new raw-input reproduction, select `"fresh"` and set the two Python interpreters and data paths. Keep code and environments in Home on Falcon, and outputs in scratch. This notebook installs nothing.

The residential Parquet is shared with [Chapter 3](https://github.com/mma4194/phd-thesis-chapter-3-synthetic-fusion/releases/tag/residential-input-v1). Obtain raw TON_IoT files from their provider; the required layout is in `README.md`.


In [ ]:
from pathlib import Path
import json, sys, tempfile
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / "chapter_support" / "reviewer.py").is_file():
    raise RuntimeError("Open this notebook from the repository root containing chapter_support/.")
sys.path.insert(0, str(ROOT))
from chapter_support.reviewer import ChapterRun

MODE = "evidence"  # "evidence" or "fresh"
SCRATCH = Path("/shared/scratch/SCWF00162")
OUTPUT = (SCRATCH if SCRATCH.is_dir() else Path(tempfile.gettempdir())) / "Chapter4_Reported_Results" / MODE
HOME = Path.home()
CONFIG = {
    "public_python": str(HOME / "venvs/thesis-public/bin/python"),
    "residential_python": str(HOME / "venvs/tiot-v5/bin/python"),
    "residential_parquet": str(HOME / "data/cps_unified_1s_FULLGRID_FINAL_WITH_IOT_STATEFUL_STRICTMASKED_TRIMMED.parquet"),
    "toniot_data_root": str(HOME / "raw_TON_IoT"),
    "workers": 4,
    "budget_hours": None,  # no application deadline; cluster allocation limits still apply
}
# A new independently executed decoder receipt can be supplied here.
# None uses the included, verified local decoder receipt.
DECODER_RECEIPT = None

review = ChapterRun(MODE, OUTPUT, CONFIG, DECODER_RECEIPT)
display(pd.DataFrame([{
    "Mode": MODE,
    "Result source": "Previously completed author runs" if MODE == "evidence" else "New raw-input computation",
    "Output folder": str(OUTPUT),
}]))


## 2. Chapter coverage

The selected stages are fixed by the chapter's experiments, not by filtering numerical rows for agreement. The comparisons retain every relevant source, seed, task and expected failure within those stages. Original TRAIN-only selection, observed-target handling, matching rules, models, seeds and numerical tolerances are retained.


In [ ]:
display(pd.read_csv(ROOT / "CHAPTER_COVERAGE.csv"))


## 3. Recorded calculations and metric responses

The arithmetic stage reconstructs published summaries from task losses, qualification records and metric-response curves. Its historical model fits are not counted as new fits. The metric-response stage runs the original canonical constructed instruments in `fresh` mode.


In [ ]:
display(review.run("records"))
display(review.run("canonical_metrics"))


## 4. Residential data and experiment

The input identity check protects the exact prepared Parquet and its observation masks. Fresh evaluation repeats the real-reference calibration, bounded and unbounded task checks, three matching plans, nine sources, five seeds and two periods. Comparisons cover 3,060 utility rows and the saved-prediction loss checks.

The original fixed candidate-task registry is an input. This workflow does not repeat the earlier exploratory discovery of that registry.


In [ ]:
display(review.run("input_verification"))
display(review.run("residential"))


In [ ]:
residential = review.table("residential", "corrected_utility_summary.csv")
primary = residential.query("matching == 'balanced_observed' and period == 'future'")
display(primary[["source", "loss_ratio", "ci_lo", "ci_hi", "outcome"]].reset_index(drop=True))


### Residential task qualifications and period effects

The primary bounded comparison qualifies three same-time groups and withholds the seven eligible temporal groups. The displayed comparisons retain both passing and withheld scientific outcomes. A withheld task or invalid generator is an expected study result, not a software error.


In [ ]:
qualification = review.table("residential", "corrected_qualification.csv")
columns = [c for c in ["suite", "capability", "eligible", "identity_ok", "relative_ok", "reference_viable", "order_ok", "final_rule_licensed"] if c in qualification]
display(qualification.query("arm == 'balanced_observed' and clipped")[columns].reset_index(drop=True))
transfer = review.table("residential", "corrected_transfer_summary.csv")
display(transfer.query("matching == 'balanced_observed'").reset_index(drop=True))


## 5. Constructed benchmark and ablations

All 18 scenarios and four base-sequence seeds are retained: 72 cases, including 52 expected-withheld cases and 20 expected-usable cases. The experiment includes the severe KT10 and milder KT18 relationship changes. These are the chapter's constructed cases used during method development.


In [ ]:
display(review.run("controlled"))
constructed = review.table("controlled", "tables/full_protocol_results.csv")
display(pd.crosstab(constructed["expected_permission"], constructed["observed_permission"], margins=True))
display(review.table("controlled", "figure_data/fig_stage_ablation_accuracy.csv"))
display(review.table("controlled", "figure_data/fig_far_frr_by_ablation.csv"))


## 6. Raw TON_IoT preparation and external evaluation

The telemetry stage verifies the 42 source members. Preparation reconstructs the five-second tables from the 15 normal captures and telemetry. Preparation integrity uses immutable prepared tables and copied input-check reports.

The evaluation includes 45 primary source runs, 27 additional-origin source runs, near and future periods, output checks, classifier resolution, and 8,508 saved-prediction loss comparisons. Expected source-invalid and insufficient-resolution outcomes are retained.


In [ ]:
display(review.run("telemetry"))
display(review.run("prepare"))
display(review.run("external"))


In [ ]:
external = review.table("external", "tables/primary_utility_summary.csv")
display(external.loc[external["period"].eq("future"), [c for c in ["source", "permission", "outcome", "loss_ratio", "lo", "hi"] if c in external]].reset_index(drop=True))
source_checks = review.table("external", "tables/source_checks.csv")
display(source_checks.groupby("source")["contract_valid"].agg(valid="sum", runs="size").reset_index())
resolution = review.table("external", "tables/primary_C2ST_summary.csv")
display(resolution.groupby("resolution_available").size().rename("source–period–feature summaries").to_frame())


## 7. Independent packet-decoder evidence

The independent local TShark execution checked the first 10,000 packets from each of the 15 hash-verified captures: 150,000 packets in total. This receipt is separate from the Falcon model execution. It covers the documented packet fields, not every payload or packet in the dataset.

To perform a new local check, follow the command in `README.md` and point `DECODER_RECEIPT` to the resulting JSON before restarting this notebook. No installation on Falcon is required for reviewing the included receipt.


In [ ]:
display(review.decoder())


## 8. Paper values beside recomputed values

All 45 registered values are recomputed from the residential task losses and external summaries selected by this run. They include loss ratios, interval endpoints, and residential direct synthetic skills. Agreement is assessed at each value's reported precision; the underlying stage comparisons also retain their original stricter numerical tolerances.


In [ ]:
headlines = review.headline_comparison()
display(headlines[["Claim", "Paper", "Recomputed", "Agreement"]])


## 9. Chapter result report

The report below covers exactly the experiments and quantities mapped above. It records the execution mode and independent decoder evidence. If a selected stage, required value or comparison fails, execution stops with the actual report location; the notebook never replaces a difference with PASS.


In [ ]:
workflow_table, chapter_report = review.finalise()
display(workflow_table)
display(chapter_report)
print("Saved chapter report:", OUTPUT / "CHAPTER_RESULTS_TO_RETURN.zip")


## Supporting material

- `README.md`: input locations, pinned environments, execution and recovery.
- `CHAPTER_COVERAGE.csv`: mapping from chapter results to executable stages.
- `supplementary/Beyond_the_Score_Supplement.pdf`: corrected thesis supplementary material.
- `protocols/paper_headlines.json`: all 45 printed-value targets.
- `evidence/`: supplied author-run tables, reports and decoder receipt.

The separate exploratory investigations remain in their original archives. They are not part of this chapter-results workflow or its PASS statement.
